# Mini-Batch Gradient Descent (MBGD) from Scratch

## Overview

This notebook implements **Mini-Batch Gradient Descent (MBGD)** from scratch using Python and NumPy. The algorithm is applied to a diabetes dataset to perform a regression task.

The implementation includes data preprocessing, feature scaling, mini-batch sampling, weight and bias updates, prediction, and model evaluation using the R² score.

**Objective:** To understand how Mini-Batch Gradient Descent optimizes a linear regression model by updating its parameters using small batches of training samples.

## 1. Import Required Libraries

- **NumPy:** Numerical computations and matrix operations.
- **Pandas:** Loading and handling the dataset.
- **Scikit-learn:** Splitting the dataset, feature scaling, and model evaluation.

In [28]:
import numpy as np
import pandas as pd

## 2. Load the Dataset

The diabetes dataset is loaded using Pandas. The `Target` column represents the dependent variable, while the remaining columns are used as independent variables.

- **Features (X):** Input variables used to make predictions.
- **Target (y):** The continuous output variable the model aims to predict.

In [29]:
df=pd.read_csv(r"C:\Users\Niloy\Downloads\ml practice\ML Algorithms\Diabetes.csv")
df

,BMI,BP,Cholesterol,LDL,Target
0,26,99,165,75,524
1,39,65,285,92,717
2,34,70,226,77,655
3,31,114,274,63,732
4,21,96,198,68,606
...,...,...,...,...,...
195,26,116,221,181,784
196,34,112,250,190,902
197,38,86,176,135,703
198,38,105,179,154,747


In [30]:
X=df.drop("Target",axis=1)
y=df["Target"]

In [31]:
X

,BMI,BP,Cholesterol,LDL
0,26,99,165,75
1,39,65,285,92
2,34,70,226,77
3,31,114,274,63
4,21,96,198,68
...,...,...,...,...
195,26,116,221,181
196,34,112,250,190
197,38,86,176,135
198,38,105,179,154


In [32]:
y

0      524
1      717
2      655
3      732
4      606
      ... 
195    784
196    902
197    703
198    747
199    804
Name: Target, Length: 200, dtype: int64

## 3. Split the Dataset

The dataset is divided into training and testing sets using `train_test_split()`.

- **Training set:** 80% of the dataset.
- **Testing set:** 20% of the dataset.
- **Random state:** 42, to make the split reproducible.

The training set is used to learn the model parameters, while the testing set evaluates performance on unseen data.

In [33]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 4. Feature Scaling Using StandardScaler

StandardScaler transforms each feature to approximately have a mean of zero and a standard deviation of one.

The scaler is fitted only on the training data to prevent data leakage. The same transformation is then applied to the testing data.

Feature scaling helps gradient-based optimization converge more effectively when the input features have different numerical scales.

In [34]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 5. Theory of Mini-Batch Gradient Descent

Mini-Batch Gradient Descent divides the training dataset into small batches and updates the model parameters after processing each batch.

### How it works

1. Initialize all weights and the bias to zero.
2. Randomly shuffle the training sample indices at the beginning of each epoch.
3. Divide the shuffled samples into mini-batches.
4. Calculate predictions for the current mini-batch.
5. Calculate the prediction errors and gradients.
6. Update the weights and bias using the learning rate.
7. Repeat the process for the specified number of epochs.

### Mathematical formulation

**Linear regression prediction:**

$$
\hat{y}=Xw+b
$$

**Mean Squared Error (MSE):**

$$
J(w,b)=\frac{1}{m}\sum_{i=1}^{m}(\hat{y}_i-y_i)^2
$$

**Gradient of the weights:**
$$
\frac{\partial J}{\partial w}
=\frac{2}{m}X^T(\hat{y}-y)
$$

**Gradient of the bias:**

$$
\frac{\partial J}{\partial b}
=\frac{2}{m}\sum_{i=1}^{m}(\hat{y}_i-y_i)
$$

**Parameter updates:**

$$
w:=w-\alpha\frac{\partial J}{\partial w}
$$

$$
b:=b-\alpha\frac{\partial J}{\partial b}
$$

Here, $(m$) is the number of samples in the current mini-batch, $(w$) represents the weights, $(b$) is the bias, and $(\alpha$) is the learning rate.

## 6. Implementation of Mini-Batch Gradient Descent

A custom Python class named `MBGD` is created to implement the algorithm without using a pre-built regression estimator.

### Class parameters

- `batch_size`: Number of training samples processed in each update.
- `learning_rate`: Controls the size of each parameter update.
- `epoch`: Number of complete passes through the training dataset.

### Methods

- `fit()`: Trains the model by updating weights and bias.
- `predict()`: Generates predictions for input features.
- `co_efficient()`: Returns the learned weights.
- `intercept()`: Returns the learned bias.

During each epoch, `np.random.permutation()` shuffles the training indices. The algorithm then processes the samples in mini-batches and updates the parameters after each batch.

**Note:** If the number of training samples is not divisible by the batch size, the final batch contains the remaining samples. The code handles this by dividing the gradients by the actual size of the current batch.

In [44]:
class MBGD:
    def __init__(self,batch_size,learning_rate,epoch):
        self.lr = learning_rate
        self.bs = batch_size
        self.epoch = epoch
        self.weight = None
        self.bias = None
    def fit(self,X_train,y_train):
        X_train = np.asarray(X_train, dtype=float)
        y_train = np.asarray(y_train, dtype=float).reshape(-1)

        self.weight = np.zeros(X_train.shape[1])
        self.bias = 0
        for epoch in range(self.epoch):
            indices = np.random.permutation(X_train.shape[0])
            for i in range(0,X_train.shape[0],self.bs):
                idx = indices[i:i+self.bs]
                y_pred=np.dot(X_train[idx],self.weight)+self.bias
                error=y_pred - y_train[idx]

                dw=(2/len(idx))*np.dot(X_train[idx].T,error)
                db=2*np.mean(error)

                self.weight-=self.lr*dw
                self.bias-=self.lr*db
    def predict(self,X_test):
        X_test=np.asarray(X_test, dtype=float)
        return np.dot(X_test,self.weight)+self.bias
    def co_efficient(self):
        return self.weight
    def intercept(self):
        return self.bias 

## 7. Model Training

The MBGD model is initialized with the following hyperparameters:

- **Batch size:** 5
- **Learning rate:** 0.01
- **Epochs:** 100

The model is trained using the scaled training features and the corresponding target values.

For each mini-batch, the algorithm calculates the predictions, computes the gradients, and updates the weights and bias.

In [75]:
mbgd=MBGD(5,0.01,100)
mbgd.fit(X_train_scaled,y_train)

## 8. Prediction on the Test Set

After training, the model predicts the target values for the scaled testing features.

The predicted values are compared with the actual target values to assess the model's ability to generalize to unseen data.

In [78]:
y_pred = mbgd.predict(X_test_scaled)
y_pred

array([705.7129487 , 761.47775826, 721.40023372, 865.97456688,
       505.41973947, 766.17892394, 741.94164527, 581.92152546,
       818.58547697, 928.2541645 , 589.48120598, 670.21799509,
       744.2223646 , 913.56405498, 650.27774096, 791.68789983,
       566.51620525, 734.17326328, 586.60149593, 534.50973207,
       798.70527181, 524.47924674, 772.99495215, 569.21180814,
       608.04579228, 598.04392804, 823.55771178, 711.82554005,
       796.57726852, 772.86941122, 667.26396967, 820.99922823,
       630.53130513, 928.63118236, 629.67094129, 663.39222954,
       832.97296436, 764.49876671, 560.60074891, 563.15336133])

## 9. Model Evaluation Using R² Score

The coefficient of determination ($(R^2$)) is used to evaluate the regression model.

$$
R^2=1-\frac{\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}
{\sum_{i=1}^{n}(y_i-\bar{y})^2}
$$

Where:

- $(y_i$): Actual target value.
- $(\hat{y}_i$): Predicted target value.
- $(\bar{y}$): Mean of the actual target values.
- $(n$): Number of testing samples.

### Interpretation

- **R² = 1:** Perfect predictions on the evaluated dataset.
- **R² = 0:** No improvement over predicting the mean of the actual test targets.
- **R² < 0:** The model performs worse than that mean-prediction baseline.

The calculated R² score indicates how well the model explains the variation in the test target values. The actual score depends on the training process and the selected hyperparameters.

In [77]:
from sklearn.metrics import r2_score
r2=r2_score(y_test,y_pred)
print("R^2",r2)

R^2 0.9480020727995707


## 10. Conclusion

In this notebook, Mini-Batch Gradient Descent was implemented from scratch using NumPy and applied to a diabetes regression dataset.

The implementation demonstrates how random shuffling, mini-batch sampling, gradient calculation, and parameter updates work together to optimize a linear regression model.

Mini-Batch Gradient Descent provides a compromise between Batch Gradient Descent, which uses the entire training dataset for each update, and Stochastic Gradient Descent, which updates parameters using one sample at a time.

The model's performance is evaluated using the R² score on the test dataset.